# DehazeNet


In [2]:
import os
import torch
import torch.nn as nn
import cv2
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image
from tqdm import tqdm
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


# model

In [7]:
class MaxoutUnit(nn.Module):
  def __init__(self, num_groups=4):
    super(MaxoutUnit, self).__init__()
    self.num_groups = num_groups

  def forward(self, x):
    batch, channels, height, width = x.size()
    x = x.view(batch, self.num_groups, -1, height, width)
    x, _ = torch.max(x, dim=1)
    return x

class DehazeNet(nn.Module):
  def __init__(self, use_brelu=True):
    super(DehazeNet, self).__init__()
    self.use_brelu = use_brelu
    self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=True)
    self.maxout1 = MaxoutUnit(num_groups=4)
    self.conv3x3 = nn.Conv2d(4, 16, kernel_size=3, padding=1, bias=True)
    self.conv5x5 = nn.Conv2d(4, 16, kernel_size=5, padding=2, bias=True)
    self.conv7x7 = nn.Conv2d(4, 16, kernel_size=7, padding=3, bias=True)
    self.maxpool3 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(48, 1, bias=True)
    self._init_weights()

  def _init_weights(self):
    for m in self.modules():
      if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
          nn.init.constant_(m.bias, 0)
      elif isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0, 0.01)
        if m.bias is not None:
          nn.init.constant_(m.bias, 0)

  def forward(self, x):
    if x.max() > 1:
      x = x / 255.0
    if x.max() > 0.5:
      x = x - 0.5

    f1 = self.conv1(x)
    f1 = self.maxout1(f1)
    f2_3x3 = self.conv3x3(f1)
    f2_5x5 = self.conv5x5(f1)
    f2_7x7 = self.conv7x7(f1)
    f2 = torch.cat([f2_3x3, f2_5x5, f2_7x7], dim=1)
    f3 = self.maxpool3(f2)
    f4 = self.avgpool(f3)
    f4 = f4.view(f4.size(0), -1)
    f4 = self.fc(f4)
    f4 = torch.clamp(f4, min=0, max=1)

    batch_size, height, width = x.size(0), x.size(2), x.size(3)
    transmission_map = f4.view(batch_size, 1, 1, 1).expand(batch_size, 1, height, width)

    return transmission_map

model = DehazeNet(use_brelu=True).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {total_params:,} parameters")

Model created with 5,857 parameters


# dataset loader

In [3]:
class DehazeDataset(Dataset):
  def __init__(self, fog_dir, clear_dir, image_size=256):
    self.fog_dir = fog_dir
    self.clear_dir = clear_dir
    self.image_size = image_size
    self.images = sorted(os.listdir(fog_dir))

    self.transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor()
    ])

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    name = self.images[idx]
    fog_path = os.path.join(self.fog_dir, name)
    clear_path = os.path.join(self.clear_dir, name)

    fog = cv2.imread(fog_path)
    clear = cv2.imread(clear_path)

    fog = cv2.cvtColor(fog, cv2.COLOR_BGR2RGB)
    clear = cv2.cvtColor(clear, cv2.COLOR_BGR2RGB)

    fog = self.transform(fog)
    clear = self.transform(clear)

    return fog, clear, name

print("DehazeDataset defined")

DehazeDataset defined


In [4]:
NUM_EPOCHS = 5
IMAGE_SIZE = 256
SAVE_DIR = "/content/drive/MyDrive/Colab Notebooks/ug_research/dehazenet/checkpoints"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/ug_research/dehazenet/outputs"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
train_dataset = DehazeDataset("/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/train/fog",
                              "/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/train/clear",
                              IMAGE_SIZE)
val_dataset = DehazeDataset("/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/val/fog",
                              "/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/val/clear",
                              IMAGE_SIZE)
test_dataset = DehazeDataset("/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/test/fog",
                              "/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/test/clear",
                              IMAGE_SIZE)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 788
Val samples: 169
Test samples: 169


# loss and optimiser

In [ ]:
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

# training

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
  model.train()
  total_loss = 0
  for fog, clear, _ in tqdm(loader, desc="Training"):
    fog = fog.to(device)
    clear = clear.to(device)

    optimizer.zero_grad()
    output = model(fog)
    loss = criterion(output, clear)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(loader)

def validate(model, loader, criterion, device):
  model.eval()
  total_loss = 0

  with torch.no_grad():
    for fog, clear, _ in tqdm(loader, desc="Validating"):
      fog = fog.to(device)
      clear = clear.to(device)

      output = model(fog)
      loss = criterion(output, clear)
      total_loss += loss.item()

  return total_loss / len(loader)

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'epoch': []}
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")

    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)

    val_loss = validate(model, val_loader, criterion, device)

    scheduler.step()

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"LR: {optimizer.param_groups[0]['lr']:.6f}")

    # save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_path = os.path.join(SAVE_DIR, f"best_model_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
        }, best_model_path)

print(f"\nBest validation loss: {best_val_loss:.4f}")


Epoch 1/5


Training:   0%|          | 0/788 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:129: UserWarning: Using a target size (torch.Size([1, 3, 256, 256])) that is different to the input size (torch.Size([1, 1, 256, 256])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
Validating: 100%|██████████| 169/169 [01:39<00:00,  1.69it/s]


Train Loss: 0.5211
Val Loss: 0.5427
LR: 0.000100

Epoch 2/5


Validating: 100%|██████████| 169/169 [00:35<00:00,  4.70it/s]


Train Loss: 0.5211
Val Loss: 0.5427
LR: 0.000100

Epoch 3/5


...

Train Loss: 0.5211
Val Loss: 0.5427
LR: 0.000050

Epoch 4/5


Validating: 100%|██████████| 169/169 [00:34<00:00,  4.94it/s]


Train Loss: 0.5211
Val Loss: 0.5427
LR: 0.000050

Epoch 5/5


Validating: 100%|██████████| 169/169 [00:35<00:00,  4.81it/s]

Train Loss: 0.5211
Val Loss: 0.5427
LR: 0.000050

Best validation loss: 0.5427


In [ ]:
# save final model
final_model_path = os.path.join(SAVE_DIR, "dehazenet_finetuned_final.pth")

torch.save({
    'model_state_dict': model.state_dict(),
    'epochs': NUM_EPOCHS,
    'final_train_loss': history['train_loss'][-1],
    'final_val_loss': history['val_loss'][-1],
    'history': history,
}, final_model_path)

print(f"\nFinal model saved to: {final_model_path}")
print(f"Model size: {os.path.getsize(final_model_path) / 1e6:.2f} MB")


Final model saved to: /content/drive/MyDrive/Colab Notebooks/ug_research/dehazenet/checkpoints/dehazenet_finetuned_final.pth
Model size: 0.03 MB


# loading best model

In [8]:
best_checkpoint = sorted([
    f for f in os.listdir(SAVE_DIR) if f.startswith('best_model')
])[-1] if any(f.startswith('best_model') for f in os.listdir(SAVE_DIR)) else None

if best_checkpoint:
    checkpoint_path = os.path.join(SAVE_DIR, best_checkpoint)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Best model loaded: {best_checkpoint}")
    print(f"Val loss: {checkpoint['val_loss']:.4f}")
else:
    print("Using final trained model for testing")

Best model loaded: best_model_epoch_4.pth
Val loss: 0.1744


# testing

## synthetic data

In [ ]:
synthetic_output_dir = os.path.join(OUTPUT_DIR, "synthetic")
os.makedirs(synthetic_output_dir, exist_ok=True)

model.eval()

with torch.no_grad():
  for fog, clear, names in tqdm(test_loader):
    fog = fog.to(device)
    output = model(fog)
    for i, name in enumerate(names):
      save_image(output[i], os.path.join(synthetic_output_dir, name))

print("Synthetic test images saved")

100%|██████████| 169/169 [04:31<00:00,  1.61s/it]

Synthetic test images saved


## real data

In [9]:
input_root = "/content/drive/MyDrive/Colab Notebooks/ug_research/real_frames"
output_root = "/content/drive/MyDrive/Colab Notebooks/ug_research/dehazenet/outputs/real"

transform = transforms.ToTensor()
model.eval()

for i in range(1, 11):
  input_dir = os.path.join(input_root, f"rf_ds{i}")
  output_dir = os.path.join(output_root, f"rf_ds{i}")

  os.makedirs(output_dir, exist_ok=True)

  img_list = os.listdir(input_dir)

  print(f"Processing rf_ds{i} with {len(img_list)} images")

  for img_name in img_list:
    img_path = os.path.join(input_dir, img_name)

    img = cv2.imread(img_path)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]
    h = h - h % 8
    w = w - w % 8
    img = img[:h, :w]

    tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
      t = model(tensor)   # transmission map [1,1,H,W]

    # reconstruct dehazed image
    A = torch.max(tensor)
    dehazed = (tensor - A * (1 - t)) / torch.clamp(t, min=0.1)
    dehazed = dehazed.clamp(0, 1)

    # convert to image
    output = dehazed.squeeze(0).permute(1, 2, 0).cpu().numpy()

    output = (output * 255).astype(np.uint8)
    output = cv2.cvtColor(output, cv2.COLOR_RGB2BGR)

    cv2.imwrite(os.path.join(output_dir, img_name), output)

  print(f"Finished rf_ds{i}")

print("All datasets processed.")

Processing rf_ds1 with 1725 images
Finished rf_ds1
Processing rf_ds2 with 735 images
Finished rf_ds2
Processing rf_ds3 with 618 images
Finished rf_ds3
Processing rf_ds4 with 1811 images
Finished rf_ds4
Processing rf_ds5 with 611 images
Finished rf_ds5
Processing rf_ds6 with 1189 images
Finished rf_ds6
Processing rf_ds7 with 2126 images
Finished rf_ds7
Processing rf_ds8 with 1183 images
Finished rf_ds8
Processing rf_ds9 with 1085 images
Finished rf_ds9
Processing rf_ds10 with 2536 images
Finished rf_ds10
All datasets processed.


# evaluation

## synthetic data

In [10]:
gt_dir = "/content/drive/MyDrive/Colab Notebooks/ug_research/dataset/test/clear"
pred_dir = "/content/drive/MyDrive/Colab Notebooks/ug_research/dehazenet/outputs/synthetic"

psnr_list = []
ssim_list = []

for name in os.listdir(gt_dir):
  gt_path = os.path.join(gt_dir, name)
  pred_path = os.path.join(pred_dir, name)

  if not os.path.exists(pred_path):
      continue

  gt = cv2.imread(gt_path)
  pred = cv2.imread(pred_path)

  gt = cv2.resize(gt, (pred.shape[1], pred.shape[0]))

  gt = cv2.cvtColor(gt, cv2.COLOR_BGR2RGB)
  pred = cv2.cvtColor(pred, cv2.COLOR_BGR2RGB)

  gt = gt.astype(np.float32) / 255.0
  pred = pred.astype(np.float32) / 255.0

  psnr = peak_signal_noise_ratio(gt, pred, data_range=1.0)
  ssim = structural_similarity(gt, pred, channel_axis=2, data_range=1.0)

  psnr_list.append(psnr)
  ssim_list.append(ssim)

print(f"PSNR: {np.mean(psnr_list):.2f}")
print(f"SSIM: {np.mean(ssim_list):.4f}")

PSNR: 13.52
SSIM: 0.5368


## real data

In [11]:
input_dir = "/content/drive/MyDrive/Colab Notebooks/ug_research/real_frames"
output_dir = "/content/drive/MyDrive/Colab Notebooks/ug_research/dehazenet/outputs/real"

results = []

for i in range(1, 11):
  input_dir = os.path.join(input_root, f"rf_ds{i}")
  output_dir = os.path.join(output_root, f"rf_ds{i}")

  sharp_input = []
  sharp_output = []
  contrast_input = []
  contrast_output = []

  img_list = [f for f in os.listdir(input_dir) if f.endswith(".jpg")]

  for name in img_list:
    in_path = os.path.join(input_dir, name)
    out_path = os.path.join(output_dir, name)
    if not os.path.exists(out_path):
        continue

    inp = cv2.imread(in_path, 0)
    out = cv2.imread(out_path, 0)
    if inp is None or out is None:
        continue

    # Sharpness
    s1 = cv2.Laplacian(inp, cv2.CV_64F).var()
    s2 = cv2.Laplacian(out, cv2.CV_64F).var()
    sharp_input.append(s1)
    sharp_output.append(s2)

    # Contrast
    c1 = np.std(inp)
    c2 = np.std(out)
    contrast_input.append(c1)
    contrast_output.append(c2)

  if len(sharp_input) == 0:
      print(f"No valid images in rf_ds{i}")
      continue

  # Averages
  s_in = np.mean(sharp_input)
  s_out = np.mean(sharp_output)

  c_in = np.mean(contrast_input)
  c_out = np.mean(contrast_output)

  print(f"\nrf_ds{i}")
  print(f"Sharpness: {s_in:.2f} -> {s_out:.2f}")
  print(f"Contrast : {c_in:.2f} -> {c_out:.2f}")

  results.append([i, s_in, s_out, c_in, c_out])


rf_ds1
Sharpness: 99.65 -> 134.31
Contrast : 62.92 -> 73.30

rf_ds2
Sharpness: 168.15 -> 193.35
Contrast : 59.65 -> 73.69

rf_ds3
Sharpness: 84.43 -> 70.57
Contrast : 62.15 -> 71.84

rf_ds4
Sharpness: 25.67 -> 20.29
Contrast : 56.60 -> 69.53

rf_ds5
Sharpness: 242.72 -> 117.79
Contrast : 57.20 -> 60.05

rf_ds6
Sharpness: 278.10 -> 231.75
Contrast : 59.71 -> 54.12

rf_ds7
Sharpness: 54.05 -> 21.30
Contrast : 47.32 -> 46.23

rf_ds8
Sharpness: 58.27 -> 102.94
Contrast : 62.32 -> 63.78

rf_ds9
Sharpness: 32.16 -> 13.08
Contrast : 48.10 -> 54.92

rf_ds10
Sharpness: 175.72 -> 211.00
Contrast : 66.75 -> 79.02
